# Milestone-3: RAG Pipeline

In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
kb = [str(row[row['answer']]) for _, row in train.iterrows()]

model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

In [ ]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
row_150 = train.iloc[150]
labels_150 = [str(row_150[l]) for l in ['A', 'B', 'C', 'D', 'E']]
res = zs(str(row_150['prompt']), candidate_labels=labels_150)
idx = res['labels'].index(str(row_150[row_150['answer']]))
print("Ground truth prob:", res['scores'][idx]) # 0.384

In [ ]:
p_emb = model.encode([str(row_150['prompt'])])
D, I = index.search(p_emb, 10) 
print("Rank of index 150:", list(I[0]).index(150) + 1) # 10

In [ ]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in I[0]]
pairs = [[str(row_150['prompt']), doc] for doc in docs_10]
scores = cross_encoder.predict(pairs)
best_idx = np.argsort(scores)[::-1]
# Rank of true doc
print("CE Rank:", list(I[0][best_idx]).index(150) + 1) # 1

In [ ]:
from transformers import AutoTokenizer
p_emb42 = model.encode([str(train.iloc[42]['prompt'])])
D42, I42 = index.search(p_emb42, 5)
docs42 = [kb[i] for i in I42[0]]
concat_docs = " ".join(docs42)
rag_str = f"Context: {concat_docs} Question: {str(train.iloc[42]['prompt'])}"
tok = AutoTokenizer.from_pretrained('bert-base-uncased')
print("Total tokens:", len(tok(rag_str)['input_ids'])) # 216

In [ ]:
rag_150 = f"Context: {kb[150]} Question: {str(row_150['prompt'])}"
res_rag = zs(rag_150, candidate_labels=labels_150)
idx_rag = res_rag['labels'].index(str(row_150[row_150['answer']]))
print("New prob:", res_rag['scores'][idx_rag]) # 0.989

In [ ]:
adv_150 = f"Context: {kb[999]} Question: {str(row_150['prompt'])}"
res_adv = zs(adv_150, candidate_labels=labels_150)
idx_adv = res_adv['labels'].index(str(row_150[row_150['answer']]))
print("Adv prob:", res_adv['scores'][idx_adv]) # 0.529